In [1]:
import import_ipynb
from preprocessing import *
from collections import Counter
import numpy as np
from first_search import get_freq_data
from sklearn.metrics.pairwise import cosine_similarity


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\obrid\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\obrid\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
100%|█████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:15<00:00, 130.58it/s]



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


100%|█████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:15<00:00, 129.11it/s]
2000it [00:00, 4921.44it/s]
2000it [00:00, 6156.00it/s]


In [14]:
en_data

,target,cleaned_text
0,"Это самая известная картина Саврасова, котора...",это самый известный картина саврасов который р...
1,"Дачу можно построить или приобрести, и тогда э...",дача построить приобрести это собственный дача...
2,В 1937-1966 гг. художественным руководителем т...,1937 1966 год. художественный руководитель теа...
3,Дети находятся в саду с утра и до возвращения ...,ребёнок находиться сад утро возвращение родите...
4,В XII−XIII вв. из дворян сформировалось сослов...,xii−xiii век. дворянин сформироваться сословие...
...,...,...
1988,Борщ обязательно входит в меню русских рестора...,борщ обязательно входить меню русский ресторан...
1989,День полного освобождения Ленинграда от фашист...,день полный освобождение ленинград фашистский ...
1990,Пушкина.,пушкин
1991,В советский период истории России слова кресть...,советский период история россия слово крестьян...


In [5]:
freq_data = get_freq_data()

2000it [00:00, 7096.39it/s]


### Частотность

In [6]:
def matrix_freq(query:str, top_k:int=5):
  clean_query = cleaner(query).split()

  query_data = dict(Counter(clean_query))

  query_vector = []

  for word in freq_data.columns:
    if word not in query_data.keys():
      query_vector.append(0.0)
    else:
      query_vector.append(query_data[word])

  query_vector = np.array(query_vector) # вектор слова

  # матрица всех частотностей
  similarities = cosine_similarity(freq_data.to_numpy(), [query_vector]).flatten()
  sorted_indices = np.argsort(similarities)[::-1]
  sorted_texts = freq_data.index[sorted_indices]

  answer_data = {'text': [], 'cosine_sim': []}
  for i in range(len(sorted_texts)):
    answer_data['text'].append(sorted_texts[i])
    answer_data['cosine_sim'].append(similarities[sorted_indices][i])

  answer_df = pd.DataFrame(answer_data)

  return answer_df.head(top_k)

### bm25

In [9]:
def matrix_bm(query:str, top_k:int=5, k=1.5, b=0.75):
  clean_query = cleaner(query).split()
  query_data = dict(Counter(clean_query))
  freq_data = get_freq_data()

  query_vector = []

  for word in freq_data.columns:
    if word not in query_data.keys():
      query_vector.append(0.0)
    else:
      query_vector.append(query_data[word])

  query_vector = np.array(query_vector)

  N = len(freq_data)
  avg_len = sum([len(text.split()) for text in en_data['cleaned_text']])/N

  #  сколько документов содержат слова для всех
  doc_freq = np.sum(freq_data > 0, axis=0)
  doc_lengths = np.sum(freq_data, axis=1).values # для каждого

  idf = np.log((N - doc_freq+0.5) / (doc_freq+0.5)) # ВОТ ТУТ, КАЖЕТСЯ, ЧТО-ТО ПОШЛО НЕ ТАК

  idf[doc_freq == 0] = 0

  # тут можно сл функцию начать
  in_query = query_vector > 0
  in_query_ind = np.where(in_query)[0]

  scores = np.zeros(N)

  for word_ind in in_query_ind:
    f = freq_data.iloc[:,word_ind].values

    if np.sum(f) > 0: # слово встречается
    
      word_idf = idf.iloc[word_ind]
        
      # тут убедись, что doc_freq нормально работает
      base_score = (f * (k+1)/ (f + k * (1 - b + b * doc_lengths/avg_len)))

      term_score = word_idf * base_score
      term_score = np.nan_to_num(term_score, 0)

      scores+=term_score


  # тут собственно сам поиск
  top_ind = np.argsort(scores)[::-1][:top_k]
  top_scores = scores[top_ind]

  answer_data = en_data[['target']].iloc[top_ind]
  answer_data['bm_sim'] = top_scores
  return answer_data.head(top_k).reset_index(drop=True)